# FinDER: Vector RAG vs Graph RAG (LanceDB + Neo4j)

This tutorial builds two parallel retrieval pipelines over the **FinDER** SEC 10-K Q&A dataset and answers the same questions through each:

1. **Vector RAG** — embed chunks, top-k similarity, augment the prompt. Backed by `LanceDBVectorStore`.
2. **Graph RAG** — entity extraction + linking, retrieve neighbors via Cypher, augment the prompt. Backed by `Neo4jGraphStore` against the bundled DozerDB instance (apoc + n10s).
3. **Hybrid** — pass both vector hits and graph context into the answer synthesizer.

**Why Neo4j for graph?** Upstream [lance-graph](https://github.com/lance-format/lance-graph/issues/91) is in development and not yet usable as a property-graph backend. For the demo we use the DozerDB instance that ships with the tutorial Docker compose. The `examples/finder/lib/lance_graph_store.py` adapter stays in the repo as a forward-compatible reference for when lance-graph stabilizes.

**Visualization.** The last section draws the constructed graph inline via NetworkX/matplotlib and points you at the **Neo4j Browser** (http://localhost:7474) for live exploration.

**Dataset.** Set `FINDER_PATH` env var to your FinDER JSON. If unset, this notebook uses `examples/finder/datasets/finder_tutorial_subset.json` so the cells run end-to-end without external data.

## Setup

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "seocho").is_dir() and (p / "examples").is_dir():
            return p
        p = p.parent
    return Path.cwd()

ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

LLM_SPEC = os.environ.get("SEOCHO_LLM", "openai/gpt-4o-mini")
LLM_PROVIDER, LLM_MODEL = (LLM_SPEC.split("/", 1) + [""])[:2]
if not LLM_MODEL:
    LLM_PROVIDER, LLM_MODEL = "openai", LLM_SPEC

PROVIDER_KEY = {
    "openai": "OPENAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "kimi": "MOONSHOT_API_KEY",
    "grok": "XAI_API_KEY",
    "qwen": "DASHSCOPE_API_KEY",
}.get(LLM_PROVIDER, "OPENAI_API_KEY")

if not os.environ.get(PROVIDER_KEY):
    raise RuntimeError(f"{PROVIDER_KEY} required for SEOCHO_LLM={LLM_SPEC}")
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is required for Tutorial 1's vector path "
        "(embeddings must use OpenAI even when SEOCHO_LLM points elsewhere)."
    )

NEO4J_URI      = os.environ.get("NEO4J_URI", "bolt://tutorials-neo4j:7687")
NEO4J_USER     = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "tutorialspw")

FINDER_PATH = os.environ.get(
    "FINDER_PATH",
    str(ROOT / "examples" / "finder" / "datasets" / "finder_tutorial_subset.json"),
)
WORKDIR = ROOT / ".seocho" / "finder_tutorial"
WORKDIR.mkdir(parents=True, exist_ok=True)
WORKSPACE_ID = "finder_tutorial"

print(f"FinDER source:  {FINDER_PATH}")
print(f"Working dir:    {WORKDIR}")
print(f"LLM:            {LLM_PROVIDER}/{LLM_MODEL}")
print(f"Embeddings:     openai/text-embedding-3-small (fixed)")
print(f"Graph backend:  Neo4j @ {NEO4J_URI}")

In [2]:
from seocho.benchmarking import load_finder_cases

cases = load_finder_cases(FINDER_PATH)
print(f"Loaded {len(cases)} FinDER cases")
for case in cases[:3]:
    print(f"  [{case.category}] {case.case_id}: {case.question}")

Loaded 10 FinDER cases
  [Company Overview] finder_tut_001: Where is Apple Inc. headquartered?
  [Financials] finder_tut_002: What was Microsoft's total revenue in fiscal year 2023?
  [Shareholder Return] finder_tut_003: How much did NVIDIA pay in dividends during fiscal year 2023?


## Vector RAG path

In [3]:
from seocho.store.vector import create_vector_store

vector_store = create_vector_store(
    kind="lancedb",
    uri=str(WORKDIR / "vec.lance"),
    table_name="finder_vec",
)

items = [
    {
        "id": case.case_id,
        "text": case.text,
        "metadata": {"category": case.category, "question": case.question},
    }
    for case in cases
]
added = vector_store.add_batch(items)
print(f"Indexed {added} chunks into LanceDB. Active rows: {vector_store.count()}")

Indexed 10 chunks into LanceDB. Active rows: 10


In [4]:
from seocho.store.llm import create_llm_backend

llm = create_llm_backend(provider=LLM_PROVIDER, model=LLM_MODEL)

ANSWER_SYSTEM = "Answer the question using only the supplied evidence. Be concise. If the evidence does not contain the answer, say so."

def answer_with_vector(question: str, *, k: int = 3) -> dict:
    t0 = time.perf_counter()
    hits = vector_store.search(question, limit=k)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    context = "\n\n".join(f"[{h.id}] {h.text}" for h in hits)
    user = f"Context:\n{context}\n\nQuestion: {question}"
    t1 = time.perf_counter()
    response = llm.complete(system=ANSWER_SYSTEM, user=user)
    gen_ms = (time.perf_counter() - t1) * 1000
    return {
        "answer": response.text.strip(),
        "retrieval_ms": retrieval_ms,
        "generation_ms": gen_ms,
        "context_chars": len(context),
        "hit_count": len(hits),
    }

vec_demo = answer_with_vector(cases[0].question)
print(f"Q: {cases[0].question}")
print(f"A: {vec_demo['answer']}")
print(f"   ({vec_demo['retrieval_ms']:.1f} ms retrieve, {vec_demo['generation_ms']:.1f} ms generate, {vec_demo['hit_count']} hits)")

Q: Where is Apple Inc. headquartered?
A: Apple Inc. is headquartered in Cupertino, California.
   (326.7 ms retrieve, 1234.6 ms generate, 3 hits)


## Graph RAG path (Neo4j)

Tiny ontology (Company / Person / FinancialMetric / Regulation, from `examples/datasets/fibo_base.jsonld`). Extraction runs through `IndexingPipeline`, writes go through seocho's standard `Neo4jGraphStore` over Bolt, and retrieval is plain Cypher — same backend and contract as production.

In [ ]:
from seocho import Ontology, Seocho
from seocho.store.graph import Neo4jGraphStore

ontology = Ontology.from_jsonld(ROOT / "examples" / "datasets" / "fibo_base.jsonld")
graph_store = Neo4jGraphStore(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

# Wipe any leftover nodes from a previous run of this tutorial.
graph_store.execute_write(
    "MATCH (n) WHERE n._workspace_id = $workspace_id DETACH DELETE n",
    params={"workspace_id": WORKSPACE_ID},
)
graph_store.ensure_constraints(ontology)

client = Seocho(
    ontology=ontology,
    graph_store=graph_store,
    llm=llm,
    workspace_id=WORKSPACE_ID,
)

for case in cases:
    client.add(case.text, metadata={"case_id": case.case_id, "category": case.category})

schema = graph_store.get_schema()
print(f"Labels:         {schema.get('labels', [])}")
print(f"Rel types:      {schema.get('relationship_types', [])}")
counts = graph_store.query(
    "MATCH (n) WHERE n._workspace_id = $workspace_id "
    "RETURN count(n) AS nodes",
    params={"workspace_id": WORKSPACE_ID},
)
edge_counts = graph_store.query(
    "MATCH ()-[r]->() WHERE r._workspace_id = $workspace_id "
    "RETURN count(r) AS rels",
    params={"workspace_id": WORKSPACE_ID},
)
print(f"Workspace size: {counts[0]['nodes']} nodes, {edge_counts[0]['rels']} rels")

In [ ]:
# Cypher-driven graph retrieval: pull seed nodes whose name matches a
# question keyword, then expand one hop. Returns a flat list of facts.
import re

STOPWORDS = {"what", "where", "who", "when", "how", "the", "a", "an", "is", "was", "were", "of", "in", "to", "for", "on", "and", "did", "during"}
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_.&\-]+")

def graph_context(question: str, *, seed_limit: int = 3, hop_limit: int = 5) -> str:
    tokens = [t for t in TOKEN_RE.findall(question) if t.lower() not in STOPWORDS]
    facts: list[str] = []
    seen: set[str] = set()
    for tok in tokens:
        seeds = graph_store.query(
            "MATCH (n) "
            "WHERE n._workspace_id = $workspace_id "
            "  AND toLower(n.name) CONTAINS toLower($kw) "
            "RETURN n.id AS id, labels(n)[0] AS label, n.name AS name, properties(n) AS props "
            "LIMIT $seed_limit",
            params={"workspace_id": WORKSPACE_ID, "kw": tok, "seed_limit": seed_limit},
        )
        for seed in seeds:
            if seed["id"] in seen:
                continue
            seen.add(seed["id"])
            facts.append(f"{seed['label']}({seed['name']}) properties={seed['props']}")
            hops = graph_store.query(
                "MATCH (n {id: $id, _workspace_id: $workspace_id})-[r]-(m) "
                "RETURN m.id AS neighbor_id, labels(m)[0] AS neighbor_label, "
                "       m.name AS neighbor_name, type(r) AS edge_type, "
                "       CASE WHEN startNode(r) = n THEN 'out' ELSE 'in' END AS direction "
                "LIMIT $hop_limit",
                params={"id": seed["id"], "workspace_id": WORKSPACE_ID, "hop_limit": hop_limit},
            )
            for hop in hops:
                arrow = "->" if hop["direction"] == "out" else "<-"
                facts.append(
                    f"{seed['name']} {arrow}[{hop['edge_type']}]{arrow} "
                    f"{hop['neighbor_label']}({hop['neighbor_name']})"
                )
    return "\n".join(facts)

demo_ctx = graph_context(cases[0].question)
print("Graph context for first question:")
print(demo_ctx or "(no matching nodes — try another question)")

In [ ]:
def answer_with_graph(question: str) -> dict:
    t0 = time.perf_counter()
    context = graph_context(question)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    if not context:
        return {
            "answer": "(no graph evidence)",
            "retrieval_ms": retrieval_ms,
            "generation_ms": 0.0,
            "context_chars": 0,
            "hit_count": 0,
        }
    user = f"Graph evidence:\n{context}\n\nQuestion: {question}"
    t1 = time.perf_counter()
    response = llm.complete(system=ANSWER_SYSTEM, user=user)
    gen_ms = (time.perf_counter() - t1) * 1000
    return {
        "answer": response.text.strip(),
        "retrieval_ms": retrieval_ms,
        "generation_ms": gen_ms,
        "context_chars": len(context),
        "hit_count": context.count("\n") + 1,
    }

graph_demo = answer_with_graph(cases[0].question)
print(f"Q: {cases[0].question}")
print(f"A: {graph_demo['answer']}")
print(f"   ({graph_demo['retrieval_ms']:.1f} ms retrieve, {graph_demo['generation_ms']:.1f} ms generate, {graph_demo['hit_count']} graph facts)")

## Hybrid: vector + graph context

In [ ]:
def answer_hybrid(question: str, *, k: int = 3) -> dict:
    t0 = time.perf_counter()
    hits = vector_store.search(question, limit=k)
    vec_ctx = "\n\n".join(f"[{h.id}] {h.text}" for h in hits)
    g_ctx = graph_context(question)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    user = (
        f"Passages:\n{vec_ctx}\n\nGraph evidence:\n{g_ctx}\n\n"
        f"Question: {question}"
    )
    t1 = time.perf_counter()
    response = llm.complete(system=ANSWER_SYSTEM, user=user)
    gen_ms = (time.perf_counter() - t1) * 1000
    return {
        "answer": response.text.strip(),
        "retrieval_ms": retrieval_ms,
        "generation_ms": gen_ms,
        "context_chars": len(vec_ctx) + len(g_ctx),
        "hit_count": len(hits) + (g_ctx.count("\n") + 1 if g_ctx else 0),
    }

hyb_demo = answer_hybrid(cases[0].question)
print(f"Q: {cases[0].question}")
print(f"A: {hyb_demo['answer']}")

## Side-by-side comparison

In [ ]:
from seocho.benchmarking import normalize_answer

def correct(answer: str, expected: str) -> bool:
    return normalize_answer(expected) in normalize_answer(answer)

rows = []
for case in cases:
    v = answer_with_vector(case.question)
    g = answer_with_graph(case.question)
    h = answer_hybrid(case.question)
    rows.append({
        "case_id": case.case_id,
        "category": case.category,
        "expected": case.expected_answer,
        "vector_answer": v["answer"],
        "vector_correct": correct(v["answer"], case.expected_answer),
        "vector_total_ms": v["retrieval_ms"] + v["generation_ms"],
        "graph_answer": g["answer"],
        "graph_correct": correct(g["answer"], case.expected_answer),
        "graph_total_ms": g["retrieval_ms"] + g["generation_ms"],
        "hybrid_answer": h["answer"],
        "hybrid_correct": correct(h["answer"], case.expected_answer),
        "hybrid_total_ms": h["retrieval_ms"] + h["generation_ms"],
    })

import pandas as pd
df = pd.DataFrame(rows)
summary = pd.DataFrame({
    "path": ["vector", "graph", "hybrid"],
    "correct_rate": [df["vector_correct"].mean(), df["graph_correct"].mean(), df["hybrid_correct"].mean()],
    "avg_total_ms": [df["vector_total_ms"].mean(), df["graph_total_ms"].mean(), df["hybrid_total_ms"].mean()],
})
print("Per-question detail:")
print(df[["case_id", "category", "vector_correct", "graph_correct", "hybrid_correct"]].to_string(index=False))
print("\nSummary:")
print(summary.to_string(index=False))

## Cleanup

Tutorial artifacts live under `.seocho/finder_tutorial/`. Delete that directory to reset.

## Visualize the constructed graph

Pull the workspace's nodes and relationships back from Neo4j and draw them inline. Open the **Neo4j Browser** at http://localhost:7474 (login `neo4j` / `tutorialspw`) for live exploration:

```cypher
MATCH (n)-[r]->(m) WHERE n._workspace_id = "finder_tutorial" RETURN n,r,m LIMIT 50
```

In [ ]:
from examples.finder.lib.graph_viz import draw_lpg, fetch_lpg_subgraph
import matplotlib.pyplot as plt

viz_nodes, viz_rels = fetch_lpg_subgraph(graph_store, workspace_id=WORKSPACE_ID, limit=80)
print(f"Drawing {len(viz_nodes)} nodes / {len(viz_rels)} relationships from Neo4j")
fig = draw_lpg(viz_nodes, viz_rels, title="FinDER graph extracted via Seocho → Neo4j")
plt.show()

In [ ]:
graph_store.close()
print(f"Artifacts: {WORKDIR}")
print("Neo4j data persists in DozerDB — drop the workspace to reset:")
print(f"  graph_store.execute_write(\"MATCH (n) WHERE n._workspace_id = '{WORKSPACE_ID}' DETACH DELETE n\")")